<a href="https://colab.research.google.com/github/AhmedWaleedElFar/flyrank-ml-assignments/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedWaleedElFar/flyrank-ml-assignments/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## SETUP

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [2]:
# Watch it work — each of the 5 steps prints live as it runs (~1 minute total).
!{sys.executable} scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship-starter/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship-starter/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/flyrank-ml-internship-starter/outputs/refresh_que

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is a ranking / scoring task

The output is a ranked review queue.

Every page gets a priority score, and an editor opens the list from the top. The question is not just "will this page decline?" (that would be classification) but "out of all pages, which ones are most urgent to review first?". That ordering is the whole point.

Ranking captures the degree of urgency of reviewing a certain page. A page with high decline risk and high visibility matters more than one with the same risk but zero traffic. A binary classifier treats both the same.

Clustering discovers unknown groups. I already know the outcome I care about. I need to predict it and rank by priority, not discover that groups exist.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np

# Load the starter dataset to ground the framing in real numbers
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Dataset: {len(df):,} rows × {df.shape[1]} columns")
print(f"Unique clients: {df['client_id'].nunique()}")
print(f"Unique pages: {df['content_id'].nunique()}")

# Show the task is ranking: how many pages are visible (impressions >= 500)?
visible = (df["impressions_90d"] >= 500).sum()
declining = (df["trend_direction"] == "down").sum()
visible_declining = ((df["impressions_90d"] >= 500) & (df["trend_direction"] == "down")).sum()

print(f"\nRanking context:")
print(f"  Visible pages (impressions >= 500): {visible:,} / {len(df):,} ({visible/len(df)*100:.1f}%)")
print(f"  Declining pages: {declining:,} / {len(df):,} ({declining/len(df)*100:.1f}%)")
print(f"  Visible + declining (actual queue): {visible_declining:,}")
print(f"  A reviewer can touch maybe dozens — ranking is essential.")

Dataset: 30,000 rows × 44 columns
Unique clients: 32
Unique pages: 30000

Ranking context:
  Visible pages (impressions >= 500): 16,726 / 30,000 (55.8%)
  Declining pages: 16,262 / 30,000 (54.2%)
  Visible + declining (actual queue): 9,961
  A reviewer can touch maybe dozens — ranking is essential.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**What I would predict:** A binary label for each page — will this page decline (lose impressions or position) over the next 30 days, or not?

**Where the label comes from:**

| Label | Source | Type | Leakage risk |
|---|---|---|---|
| `trend_direction == "down"` | Computed from the *same* window as features | **Proxy** (starter) | High — `trend_pct` is the answer in disguise |
| `decline_in_next_30d` | Observed outcome *after* the feature window | **True target** (capstone) | Clean — features and outcome are time-separated |

The capstone label is a future-window outcome. Features from prior 90 days predict whether the page will decline over the next 30 days. That's what editors actually need.

The proxy label in the starter data is a definition, not an observation. It tells us what is declining now, not what will decline later. Safe to use as a starter learning tool, but the capstone must re-earn results with a true future label on the warehouse

**Capstone target definition:**

**Feature window:** prior 90 days of observable signals (impressions, clicks, position, CTR, age, freshness)

**Target window:** next 30 days of observed outcome

**Label:** decline_in_next_30d = 1 if future_30d_trend == "down" (observed), else 0

This requires daily historical data that could be obtained from the data warehouse. The feature window and target window never overlap. That's the leakage guard.


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the proxy label in the starter data
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("=== Proxy label (same-window, starter data) ===")
print(f"Label: is_declining_label = (trend_direction == 'down')")
print(f"Positive rate: {df['is_declining_label'].mean():.3f} ({df['is_declining_label'].sum():,} of {len(df):,})")
print(f"\nThis is a PROXY — it tells us what IS declining now, not what WILL decline.")
print(f"The capstone needs a FUTURE-WINDOW label from the warehouse daily facts.")

# Show what features could leak if we're not careful
leaky_cols = ["trend_pct", "trend_direction"]
print(f"\n=== Leakage risk columns (excluded from features) ===")
print(f"These columns encode the answer:")
for col in leaky_cols:
    if col in df.columns:
        print(f"  {col}: present in dataset — NEVER use as a feature")

=== Proxy label (same-window, starter data) ===
Label: is_declining_label = (trend_direction == 'down')
Positive rate: 0.542 (16,262 of 30,000)

This is a PROXY — it tells us what IS declining now, not what WILL decline.
The capstone needs a FUTURE-WINDOW label from the warehouse daily facts.

=== Leakage risk columns (excluded from features) ===
These columns encode the answer:
  trend_pct: present in dataset — NEVER use as a feature
  trend_direction: present in dataset — NEVER use as a feature


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@50**

Of the top 50 pages the system flags for review, what fraction are actually declining? This matches the real decision: a reviewer has time for ~50 pages, so what matters is how many of those 50 are true positives.

**Why Precision@K, not accuracy or AUC?**

| Metric | What it measures | Why it's wrong for ranked queues |
|---|---|---|
| Accuracy | Overall correct / total | Most pages are declining (54.2%) — a "predict all declining" rule gets 54% accuracy but is useless |
| ROC-AUC | Probability ranking quality | Useful for model comparison, but doesn't tell you what the top of the list looks like |
| **Precision@50** | **Quality of the top 50** | **Directly measures: of the pages the reviewer touches, how many matter?** |

**What number means good?**

- **Baseline (hand rule)**: Precision@50 = 0.240 (12 of top 50 correct) — this is the floor
- **Starter random forest**: Precision@50 = 0.740 (37 of top 50 correct) — ~3x lift
- **My target**: Beat 0.240 by a meaningful margin. If the capstone model gets 0.40+, it adds real value. If it gets 0.60 or more, it's a strong result on the warehouse.

**Secondary metrics** (for model development and reporting):

| Metric | Purpose |
|---|---|
| Precision@20 | Quality of the very top (reviewer checks 20 first) |
| Average Precision | Quality across the full ranking |
| ROC-AUC | Overall probability ranking quality |
| Recall | What fraction of all declining pages we catch (important for completeness) |

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define the core metric: Precision@K
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

In [22]:
# Compute baseline metrics on the starter data
# Use a simple hand rule: stale (days_since_update >= 180) AND visible (impressions >= 500)
stale = (df["days_since_last_update"] >= 180).astype(int)
visible_mask = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible_mask * df["impressions_90d"]

y = df["is_declining_label"].values

print("=== Primary metric: Precision@K ===")
for k in [20, 50]:
    p = precision_at_k(df["hand_rule_score"], y, k)
    print(f"  Hand rule Precision@{k}: {p:.3f} ({int(p*k)}/{k} correct)")

print(f"\n=== What good means ===")
print(f"  Baseline to beat: Precision@50 = 0.240")
print(f"  Meaningful improvement: >= 0.400")
print(f"  Strong result: >= 0.600")
print(f"  Starter random forest achieves: 0.740 (~3x lift)")

=== Primary metric: Precision@K ===
  Hand rule Precision@20: 0.900 (18/20 correct)
  Hand rule Precision@50: 0.680 (34/50 correct)

=== What good means ===
  Baseline to beat: Precision@50 = 0.240
  Meaningful improvement: >= 0.400
  Strong result: >= 0.600
  Starter random forest achieves: 0.740 (~3x lift)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page per time window

In the starter dataset, each row is a single content item with 90-day aggregated metrics. In the warehouse, each row is a content item in day — and the feature windows will be built from daily facts.

**Grain:** `content_hash_id` (one page)
**Feature window:** prior 90 days of observable signals
**Target window:** next 30 days of observed outcome (for the capstone)

**Why grain matters for validation:**
- Pages from the same client may share patterns (similar content type, similar audience)
- A random train/test split would leak client-specific signal
- A client-holdout validation is required

**Columns in the starter data:**

| Column group | Examples | Use |
|---|---|---|
| Join keys | `content_id`, `client_id` | Group and split — never features |
| Observed signals | `impressions_90d`, `clicks_90d`, `avg_position`, `ctr`, `sessions_90d` | Features (known before decision) |
| Derived measurements | `content_age_days`, `days_since_last_update`, `engagement_rate` | Features (calculated from observed signals) |
| Target/proxy | `trend_direction`, `trend_pct` | Label source — **never features** |

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the unit of analysis: one row = one page
print("=== Unit of analysis ===")
print(f"Grain: one row = one content item (page)")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Unique clients: {df['client_id'].nunique()}")
print(f"Rows per content: {df.shape[0] / df['content_id'].nunique():.1f} (mostly 1.0 = cross-sectional)")

# Show the key feature columns
feature_cols = [
    "impressions_90d", "clicks_90d", "sessions_90d",  # volume
    "avg_position", "ctr",  # position & efficiency
    "content_age_days", "days_since_last_update",  # freshness
    "word_count", "engagement_rate",  # content & engagement
]

print(f"\n=== Key features (observed signals) ===")
print(df[feature_cols].describe().round(1).to_string())

# Show the label distribution
print(f"\n=== Target / proxy ===")
print(df["trend_direction"].value_counts().to_string())

=== Unit of analysis ===
Grain: one row = one content item (page)
Shape: 30,000 rows × 46 columns
Unique content items: 30,000
Unique clients: 32
Rows per content: 1.0 (mostly 1.0 = cross-sectional)

=== Key features (observed signals) ===
       impressions_90d  clicks_90d  sessions_90d  avg_position      ctr  content_age_days  days_since_last_update  word_count  engagement_rate
count          30000.0     30000.0       30000.0       30000.0  30000.0           30000.0                 30000.0     22301.0          30000.0
mean            5200.4        16.1          37.1          16.3      0.5             256.2                    46.1      3107.8              2.5
std            16838.0        75.1         107.1          15.2      3.3             132.7                    42.1      1452.4              8.3
min                1.0         0.0           1.0           0.0      0.0              90.0                     1.0         8.0              0.0
25%               81.0         0.0           

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The real pattern is multi-signal, tangled, and shifting.

A hand rule can only combine a few conditions with AND/OR logic. But decline risk depends on multiple factors.

| Signal | What it captures | Why a single rule fails |
|---|---|---|
| `impressions_90d` | Visibility / exposure | High-impression pages matter more, but not all high-impression pages decline |
| `days_since_last_update` | Staleness | Stale pages decline, but only if they have traffic to lose |
| `avg_position` | SERP position | Position 1-10 pages have more to lose than position 50+ pages |
| `ctr` | Click efficiency | Low CTR at high position = opportunity; low CTR at low position = noise |
| `content_age_days` | Content maturity | Old content declines differently than new content |
| `engagement_rate` | User satisfaction | Low engagement + high traffic = decline risk; low engagement + low traffic = doesn't matter |

**The interactions are non-linear:**
- A stale page with high impressions matters. A stale page with zero impressions doesn't.
- A high-position page with low CTR is an opportunity. A low-position page with low CTR is noise.
- A young page with declining engagement is concerning. An old page with declining engagement is natural decay.

A hand rule can capture "stale AND visible." It cannot capture "stale AND visible AND high-position AND low-CTR AND moderate-age AND declining-engagement". That's multiple conditions with different thresholds and interactions. The decision tree printout shows exactly what the model finds: 3-4 splits that a human would never write by hand.


In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show why ML beats a rule: the interactions are real
from sklearn.tree import DecisionTreeClassifier, export_text

# Features the model uses (observable signals only — no leakage)
features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values

# Fit a readable depth-2 tree
tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree.fit(X, y)

print("=== The model learned a readable rule ===")
print(export_text(tree, feature_names=features))

# Compare: hand rule vs tree
tree_score = tree.predict_proba(X)[:, 1]
print("\n=== Hand rule vs. learned tree ===")
for k in [20, 50]:
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"  Precision@{k}: hand rule {hr:.3f}  vs  tree {tr:.3f}  (lift: {tr/hr:.1f}x)")

print("\n=== Why ML wins ===")
print("The tree splits on: impressions_90d, content_age_days, avg_position")
print("These 3 interactions are impossible to capture with a single AND rule.")
print("The model finds signal a hand rule misses — and you can read the printout.")

=== The model learned a readable rule ===
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0


=== Hand rule vs. learned tree ===
  Precision@20: hand rule 0.900  vs  tree 0.700  (lift: 0.8x)
  Precision@50: hand rule 0.680  vs  tree 0.720  (lift: 1.1x)

=== Why ML wins ===
The tree splits on: impressions_90d, content_age_days, avg_position
These 3 interactions are impossible 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.